# Tendril VAE Tutorial

**Purpose**: Demonstrate how to train and verify a Tendril VAE using the candescence package  
**Author**: Hallett Lab  
**Date**: 2026-01-27

This notebook walks through the complete workflow:
1. Environment setup and imports
2. Configuration with TLVConfig
3. Dataset loading and exploration
4. Model preparation
5. Training (optional - can use pre-trained)
6. Loading a trained model
7. Inference and verification
8. Latent space analysis

## Environment

```bash
uv sync   # or, once created: source .venv/bin/activate
```

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path for development (not needed if package is installed)
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Verify imports work
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Enable inline plotting for Jupyter/VSCode
%matplotlib inline

# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Candescence imports
from candescence.core.config import TLVConfig
from candescence.core.logging_config import get_logger, configure_logging
from candescence.tlv.factory import Factory
from candescence.tlv.data import FullDataset
from candescence.tlv.architectures import tendril_VAE
from candescence.tlv.inference import Inference, LatentEmbedding

# Setup logging
configure_logging(level='INFO')
logger = get_logger("tutorial")

print("All imports successful!")

## 2. Configuration

The `TLVConfig` class manages all experiment paths and parameters. It automatically creates the required directory structure.

In [ ]:
# Create configuration for a tutorial experiment
# This will create directories under <refined>/

config = TLVConfig(
    experiment_name="tutorial_tendril_vae",
    save_name="test_run_v1"
)

# Create the experiment directories
config.create_directories()

print(f"Experiment path: {config.exp_path}")
print(f"Models path: {config.models_path}")
print(f"Analyses path: {config.analyses_path}")
print(f"Loss statistics path: {config.loss_path}")

In [ ]:
# Configure model architecture - Tendril VAE (Strategy 14)

# Architecture settings
config.strategy = 14
config.architecture = 'tendril_vae'
config.latent_dim = 128
config.intermediate_dim = 256
config.kernel_size = 3
config.leaky_relu_slope = 0.02

# Conditioning - Tendril VAE uses hue as conditional variable
# Encoder sees actual hue, decoder gets fixed value (0.5) to learn hue-invariant features
config.conditional_variables = ['average_hue']
config.conditional_decoder_fixed_values = {'average_hue': 0.5}
config.augment_decoder_images = False  # Use fixed decoder values, not augmented lookup
config.adjust_images = False

# Dataset settings
config.image_dimension = 128
config.image_type = 'non-wash'
config.restrict_to_day = 2
config.dataset_seed = 9954
config.training_seed = 9954

# Full training dataset sizes
config.train_num = 1200
config.validation_num = 400
config.test_num = 400

# Full training settings
config.batch_size = 256
config.number_epochs = 40
config.vae_lr = 1e-4
config.film_lr = 5e-4
config.weight_decay = 1.5e-3

# Loss weights
config.kl_weight = 1
config.LPIPS_weight = 10
config.MSE_weight = 100
config.skip_weight = 1
config.conditional_loss_weight = 1000

# System settings
config.process_priority = 19
config.num_threads = 10

print("Configuration complete!")
print(f"Strategy: {config.strategy}")
print(f"Architecture: {config.architecture}")
print(f"Latent dim: {config.latent_dim}")
print(f"Conditional variables: {config.conditional_variables}")
print(f"Training samples: {config.train_num}")
print(f"Epochs: {config.number_epochs}")
print(f"Batch size: {config.batch_size}")

## 3. Dataset Loading

The `FullDataset` class handles image loading, preprocessing, and train/validation/test splits.

In [ ]:
# Create the Factory which manages the experiment
factory = Factory(config)

# Load dataset
factory.load_dataset()

print(f"\nDataset loaded!")
print(f"Total samples: {len(factory.dataset)}")
print(f"Training samples: {len(factory.dataset.train_dataset)}")
print(f"Validation samples: {len(factory.dataset.validation_dataset)}")
print(f"Test samples: {len(factory.dataset.test_dataset)}")

In [ ]:
# Explore the dataset
target_df = factory.dataset.target_df
print(f"\nDataFrame shape: {target_df.shape}")
print(f"\nColumns: {list(target_df.columns)[:15]}...")  # First 15 columns
print(f"\nSample data:")
target_df[['id', 'average_hue', 'average_saturation', 'average_value', 'colony_size']].head()

In [ ]:
# Visualize sample images from the dataset
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, ax in enumerate(axes.flatten()):
    if i < len(target_df):
        img = target_df.iloc[i]['rgb_image']
        hue = target_df.iloc[i]['average_hue']
        ax.imshow(img)
        ax.set_title(f"Hue: {hue:.3f}", fontsize=10)
        ax.axis('off')

plt.suptitle("Sample Images from Dataset", fontsize=14)
plt.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# Distribution of conditional variable (hue)
# Note: Hue values are stored in 0-255 range (PIL HSV mode)
# The decoder sees hue normalized to 0-1 range (config value 0.5 = 127.5 in storage scale)
decoder_fixed_hue = config.conditional_decoder_fixed_values.get('average_hue', 0.5)
decoder_fixed_hue_storage = decoder_fixed_hue * 255  # Convert to storage scale

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(target_df['average_hue'], bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Average Hue (storage scale: 0-255)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Hue Values in Dataset')
ax.axvline(x=decoder_fixed_hue_storage, color='red', linestyle='--', 
           label=f'Decoder fixed value ({decoder_fixed_hue:.1f} normalized = {decoder_fixed_hue_storage:.1f} storage)')
ax.legend()
plt.tight_layout()
display(fig)
plt.close(fig)

## 4. Model Preparation

Prepare the Tendril VAE model with separate optimizers for VAE and FiLM parameters.

In [ ]:
# Prepare VAE model
factory.prepare_vae()

print(f"\nModel architecture: {type(factory.vae).__name__}")
print(f"Device: {config.device}")

# Count parameters
total_params = sum(p.numel() for p in factory.vae.parameters())
trainable_params = sum(p.numel() for p in factory.vae.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# Inspect model structure
print("Encoder structure:")
print(factory.vae.encoder)
print("\n" + "="*50 + "\n")
print("Decoder structure:")
print(factory.vae.decoder)

In [ ]:
# Setup dataloaders
factory.set_training_dataloader()

print(f"Training batches: {len(factory.train_dataloader)}")
print(f"Validation batches: {len(factory.validation_dataloader)}")
print(f"Batch size: {config.batch_size}")

## 5. Training (Optional)

Train the model from scratch. For production use, increase `number_epochs` to 100.

**Note**: Training takes significant time. For this tutorial, we use few epochs. Skip this section if loading a pre-trained model.

In [ ]:
# Train from scratch (full training mode)
print("Starting training...")
factory.train_model()
print("Training complete!")

In [ ]:
# Option B: Load pre-trained model (SKIP if training from scratch above)
# Uncomment below to load pre-trained weights instead of training

# pretrained_path = Path("<refined>/0_37_tendrils_reproduction/manuscript_v1/models/model.pth")
# 
# if pretrained_path.exists():
#     print(f"Loading pre-trained model from: {pretrained_path}")
#     state = torch.load(pretrained_path, map_location=config.device, weights_only=True)
#     factory.vae.load_state_dict(state)
#     factory.vae.to(config.device)
#     print("Model loaded successfully!")
# else:
#     print(f"Pre-trained model not found at {pretrained_path}")

print("Skipping pre-trained model loading - using freshly trained model")

## 6. Verification: Forward Pass Test

Verify the model works correctly by running a forward pass.

In [ ]:
# Test forward pass
factory.vae.eval()

# Get a sample batch
sample_batch = next(iter(factory.validation_dataloader))

# For tendril VAE (strategy 14): batch contains (image, cond_encoder, cond_decoder, target)
img, cond_enc, cond_dec, target = sample_batch

print(f"Input image shape: {img.shape}")
print(f"Encoder condition shape: {cond_enc.shape}")
print(f"Decoder condition shape: {cond_dec.shape}")

# Move to device
img = img.to(config.device)
cond_enc = cond_enc.to(config.device).float()
cond_dec = cond_dec.to(config.device).float()

# Forward pass through encoder
with torch.no_grad():
    z, mu, logvar, skip = factory.vae.encoder(img, cond_enc)
    print(f"\nLatent z shape: {z.shape}")
    print(f"Mu shape: {mu.shape}")
    print(f"Logvar shape: {logvar.shape}")
    print(f"Skip connections: {len(skip)} tensors")
    
    # Forward pass through decoder
    reconstruction = factory.vae.decoder(z, skip, cond_dec)
    print(f"\nReconstruction shape: {reconstruction.shape}")

print("\nForward pass successful!")

In [ ]:
# Visualize reconstruction
from candescence.tlv.utilities import convert_rgb_transformed_2_rgb

# Select a few samples to visualize
n_samples = 5

fig, axes = plt.subplots(2, n_samples, figsize=(3*n_samples, 6))

for i in range(n_samples):
    # Original
    orig_img = convert_rgb_transformed_2_rgb(img[i].cpu())
    axes[0, i].imshow(orig_img)
    axes[0, i].set_title(f"Original\nHue: {cond_enc[i, 0].item():.3f}")
    axes[0, i].axis('off')
    
    # Reconstruction
    rec_img = convert_rgb_transformed_2_rgb(reconstruction[i].cpu())
    axes[1, i].imshow(rec_img)
    axes[1, i].set_title(f"Reconstruction\nDec cond: {cond_dec[i, 0].item():.3f}")
    axes[1, i].axis('off')

plt.suptitle("Original vs Reconstruction", fontsize=14)
plt.tight_layout()
display(fig)
plt.close(fig)

## 7. Latent Space Analysis

Analyze the latent space learned by the VAE.

In [ ]:
# Collect latent embeddings for all validation samples
factory.vae.eval()

all_z = []
all_hue = []

with torch.no_grad():
    for batch in factory.validation_dataloader:
        img, cond_enc, cond_dec, _ = batch
        img = img.to(config.device)
        cond_enc = cond_enc.to(config.device).float()
        
        z, mu, logvar, skip = factory.vae.encoder(img, cond_enc)
        all_z.append(mu.cpu().numpy())  # Use mu for deterministic embedding
        all_hue.append(cond_enc[:, 0].cpu().numpy())

all_z = np.concatenate(all_z, axis=0)
all_hue = np.concatenate(all_hue, axis=0)

print(f"Collected {len(all_z)} latent embeddings")
print(f"Latent shape: {all_z.shape}")

In [ ]:
# PCA visualization of latent space
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
z_pca = pca.fit_transform(all_z)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(z_pca[:, 0], z_pca[:, 1], c=all_hue, cmap='hsv', s=30, alpha=0.7)
plt.colorbar(scatter, ax=ax, label='Hue')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA of Latent Space (colored by Hue)')
plt.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# Latent dimension statistics
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Mean per dimension
axes[0, 0].bar(range(all_z.shape[1]), all_z.mean(axis=0))
axes[0, 0].set_xlabel('Latent Dimension')
axes[0, 0].set_ylabel('Mean')
axes[0, 0].set_title('Mean of Each Latent Dimension')

# Std per dimension
axes[0, 1].bar(range(all_z.shape[1]), all_z.std(axis=0))
axes[0, 1].set_xlabel('Latent Dimension')
axes[0, 1].set_ylabel('Std')
axes[0, 1].set_title('Std of Each Latent Dimension')

# Correlation with hue for each dimension
correlations = [np.corrcoef(all_z[:, i], all_hue)[0, 1] for i in range(all_z.shape[1])]
axes[1, 0].bar(range(len(correlations)), correlations)
axes[1, 0].set_xlabel('Latent Dimension')
axes[1, 0].set_ylabel('Correlation with Hue')
axes[1, 0].set_title('Correlation of Each Latent Dimension with Hue')
axes[1, 0].axhline(y=0, color='red', linestyle='--', alpha=0.5)

# Distribution of a high-correlation dimension
top_corr_dim = np.argmax(np.abs(correlations))
axes[1, 1].scatter(all_z[:, top_corr_dim], all_hue, alpha=0.5, s=10)
axes[1, 1].set_xlabel(f'Latent Dim {top_corr_dim}')
axes[1, 1].set_ylabel('Hue')
axes[1, 1].set_title(f'Highest Correlated Dim ({top_corr_dim}): r={correlations[top_corr_dim]:.3f}')

plt.tight_layout()
display(fig)
plt.close(fig)

## 8. Latent Walk Visualization

Visualize how changing specific latent dimensions affects the reconstruction.

In [ ]:
# Select a sample image for latent walk
sample_idx = 0
sample_batch = next(iter(factory.validation_dataloader))
img, cond_enc, cond_dec, _ = sample_batch

img_single = img[sample_idx:sample_idx+1].to(config.device)
cond_enc_single = cond_enc[sample_idx:sample_idx+1].to(config.device).float()
cond_dec_single = cond_dec[sample_idx:sample_idx+1].to(config.device).float()

# Encode
with torch.no_grad():
    z, mu, logvar, skip = factory.vae.encoder(img_single, cond_enc_single)

print(f"Original hue: {cond_enc_single[0, 0].item():.3f}")
print(f"Latent z shape: {z.shape}")

In [ ]:
# Latent walk: vary specific dimensions
dims_to_walk = [0, top_corr_dim, 10, 50]  # Include the dimension most correlated with hue
n_steps = 7
walk_range = np.linspace(-3, 3, n_steps)

fig, axes = plt.subplots(len(dims_to_walk) + 1, n_steps, figsize=(2*n_steps, 2*(len(dims_to_walk)+1)))

# Original image in first row
orig_img = convert_rgb_transformed_2_rgb(img_single[0].cpu())
for j in range(n_steps):
    if j == n_steps // 2:
        axes[0, j].imshow(orig_img)
        axes[0, j].set_title("Original")
    else:
        axes[0, j].axis('off')
    axes[0, j].axis('off')

# Latent walks
for i, dim in enumerate(dims_to_walk):
    for j, alpha in enumerate(walk_range):
        z_mod = z.clone()
        z_mod[0, dim] = z[0, dim] + alpha
        
        with torch.no_grad():
            rec = factory.vae.decoder(z_mod, skip, cond_dec_single)
        
        rec_img = convert_rgb_transformed_2_rgb(rec[0].cpu())
        axes[i+1, j].imshow(rec_img)
        if j == 0:
            axes[i+1, j].set_ylabel(f"Dim {dim}", fontsize=10)
        if i == 0:
            axes[i+1, j].set_title(f"{alpha:.1f}", fontsize=9)
        axes[i+1, j].axis('off')

plt.suptitle("Latent Walk: Varying Individual Dimensions", fontsize=14)
plt.tight_layout()
display(fig)
plt.close(fig)

## 9. Hue Conditioning Test

Test the conditional generation by varying the decoder hue condition.

In [ ]:
# Vary decoder conditioning (hue) while keeping latent fixed
hue_values = np.linspace(0.0, 1.0, 7)

fig, axes = plt.subplots(2, len(hue_values), figsize=(2.5*len(hue_values), 5))

# Row 1: Same latent, different decoder hue
for j, hue in enumerate(hue_values):
    cond_dec_mod = torch.tensor([[hue]], device=config.device, dtype=torch.float32)
    
    with torch.no_grad():
        rec = factory.vae.decoder(z, skip, cond_dec_mod)
    
    rec_img = convert_rgb_transformed_2_rgb(rec[0].cpu())
    axes[0, j].imshow(rec_img)
    axes[0, j].set_title(f"Hue: {hue:.2f}")
    axes[0, j].axis('off')

# Row 2: Original and info
axes[1, 0].imshow(orig_img)
axes[1, 0].set_title("Original")
axes[1, 0].axis('off')

for j in range(1, len(hue_values)):
    axes[1, j].axis('off')

axes[1, 3].text(0.5, 0.5, f"Original Hue: {cond_enc_single[0, 0].item():.3f}", 
                ha='center', va='center', fontsize=12, transform=axes[1, 3].transAxes)

plt.suptitle("Decoder Hue Conditioning: Same Latent, Different Hue", fontsize=14)
plt.tight_layout()
display(fig)
plt.close(fig)

## 10. Summary

This tutorial demonstrated:

1. **Configuration**: Using `TLVConfig` to manage experiment paths and parameters
2. **Dataset**: Loading and exploring the Candida colony image dataset
3. **Model**: Preparing the Tendril VAE architecture with FiLM conditioning
4. **Training**: Running the training loop (or loading pre-trained weights)
5. **Verification**: Testing forward pass and visualizing reconstructions
6. **Analysis**: Exploring the latent space structure and correlations
7. **Latent Walk**: Visualizing the effect of individual latent dimensions
8. **Conditioning**: Testing the hue conditioning mechanism

### Next Steps

- Run full training with `number_epochs=100` and larger dataset
- Use `Inference` class for comprehensive analysis
- Generate latent space clustering analysis
- Create manuscript figures with the production model

In [ ]:
# Save configuration summary
print("\n" + "="*50)
print("CONFIGURATION SUMMARY")
print("="*50)
summary = factory.get_config_summary()
for key, value in summary.items():
    print(f"{key}: {value}")